# 16 - Multithreading


## Free-Threaded Python (`3.13t` & `3.14t` - No-GIL)

PEP 703 kapsamında Python 3.13 ile isteğe bağlı deneysel olarak sunulan ve Python 3.14 ile resmi destek kazanan **free-threaded** derlemeler (`3.13t` / `3.14t`), Global Interpreter Lock (GIL) olmadan birden fazla thread'in tüm CPU çekirdeklerinde gerçek paralel olarak çalışmasına olanak tanır.

***Önemli not:*** Daha resmi olarak yayınlanmasa da python resmi olarak tam Multithreading desteği ile güncellenmektedir.  

[Video Linki: https://www.youtube.com/shorts/yrvXn4l-Brc](https://www.youtube.com/shorts/yrvXn4l-Brc)

**Kurulum ve Kontrol**

```bash
uv python install 3.14t
python3.14t -VV
```

**Python içinde kontrol betiği**

```Python
import sys
import sysconfig

gil_disabled_build: bool = bool(sysconfig.get_config_var('Py_GIL_DISABLED'))
gil_active: bool = sys._is_gil_enabled() if hasattr(sys, '_is_gil_enabled') else True

print(f'Python version: {sys.version.split()[0]}')
print(f'Free-threaded build: {gil_disabled_build}')
print(f'GIL currently enabled: {gil_active}')
```


## 1. Threads, GIL, Concurrency vs. Parallelism

`threading` modülü, aynı süreç (process) içinde birden fazla iş parçacığı (thread) oluşturarak özellikle ağ istekleri ve dosya işlemleri gibi G/Ç (I/O) beklemesi içeren görevleri eşzamanlı yürütmeyi sağlar[cite: 74].

**Syntax**

```python
import threading

t = threading.Thread(target=worker_fn, args=(arg1,), kwargs={'key': val})
t.start()
t.join()
```

**Temel Kavramlar ve Dikkat Edilmesi Gerekenler**
* **GIL (Global Interpreter Lock):** Standart CPython yorumlayıcısında aynı anda yalnızca tek bir thread'in Python bytecode çalıştırmasına izin veren kilittir[cite: 74]. *(Not: Python 3.13 ile deneysel olarak tanıtılan ve Python 3.14 ile resmi olarak desteklenen isteğe bağlı `free-threaded` (`3.14t` / `--disable-gil`) derlemelerinde GIL devre dışı bırakılabilir; ancak varsayılan Python derlemelerinde GIL aktiftir.)*
* **Concurrency vs. Parallelism:** `asyncio` ve standart `threading` eşzamanlılık (concurrency) sağlar; görevler tek çekirdek üzerinde sırayla geçiş yaparak (context switching) ilerler[cite: 74]. Gerçek CPU paralelliği (parallelism) için çok çekirdekli işlemcilerde `multiprocessing` (veya free-threaded Python) kullanılır[cite: 74].
* **`asyncio` vs. `threading`:** `asyncio` tek thread üzerinde çalışır ve kontrol geçişleri geliştiricinin yazdığı `await` noktalarında deterministik olarak gerçekleşir; `threading` ise birden fazla işletim sistemi thread'i oluşturur ve thread geçiş zamanlaması işletim sistemi (OS scheduler) tarafından belirlenir[cite: 74].
* **`start()` ve `join()`:** `t.start()` thread'i başlatır; `t.join()` ise çağıran ana thread'in ilgili thread tamamlanana kadar beklemesini sağlar[cite: 74].


In [ ]:
import threading
import time


def worker(label: str, steps: int, delay: float) -> None:
    print(f'Starting {label}')
    for step in range(1, steps + 1):
        time.sleep(delay)
        print(f'{label}: {step}')


t1: threading.Thread = threading.Thread(target=worker, args=('Worker-A', 3, 0.2))
t2: threading.Thread = threading.Thread(target=worker, kwargs={'label': 'Worker-B', 'steps': 3, 'delay': 0.15})

t1.start()
t2.start()

t1.join()
t2.join()
print('All workers completed.')


## 2. Thread Synchronization with `threading.Lock`

`threading.Lock`, birden fazla thread'in kritik bir kod bloğuna veya paylaşılan bir kaynağa aynı anda erişmesini engelleyen karşılıklı dışlama (mutual exclusion / mutex) kilididir[cite: 75].

**Syntax**

```python
lock = threading.Lock()
lock.acquire()
try:
    critical_section()
finally:
    lock.release()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Sıralı Erişim:** Kilidi `acquire()` ile alan thread işi bitip `release()` çağırana kadar aynı kilidi bekleyen diğer thread'ler bloklanır[cite: 75].
* **Kilit Kapsamı:** Kilit yalnızca o `Lock` nesnesini `acquire()` eden thread'leri koordine eder; kilidi kullanmayan bağımsız thread'ler paralel şekilde çalışmaya devam eder[cite: 75].
* **Deadlock Riski:** Alınan bir kilidin istisna oluşması veya unutulma nedeniyle `release()` edilmemesi, kilidi bekleyen diğer tüm thread'lerin sonsuza dek kilitlenmesine (deadlock) yol açar[cite: 75].


In [ ]:
import threading
import time

lock: threading.Lock = threading.Lock()


def exclusive_job(name: str, count: int) -> None:
    lock.acquire()
    try:
        for i in range(1, count + 1):
            time.sleep(0.1)
            print(f'{name}: {i}')
    finally:
        lock.release()


t1: threading.Thread = threading.Thread(target=exclusive_job, args=('Locked-1', 3))
t2: threading.Thread = threading.Thread(target=exclusive_job, args=('Locked-2', 3))

t1.start()
t2.start()
t1.join()
t2.join()


## 3. Daemon Threads (`daemon=True`)

Daemon thread'ler, arka planda düşük öncelikli yardımcı görevler (loglama, periyodik durum kontrolü, önbellek temizliği vb.) yürüten ve ana programın kapanmasını engellemeyen iş parçacıklarıdır[cite: 76].

**Syntax**
```python
bg_thread = threading.Thread(target=background_task, daemon=True)
bg_thread.start()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Sonlanma Davranışı:** Python süreci, aktif olan tüm **non-daemon** (normal) thread'ler tamamlandığında programdan çıkış yapar; geriye yalnızca `daemon=True` olan thread'ler kalmışsa bu thread'ler anında sonlandırılır[cite: 76].
* **Veri Bütünlüğü Uyarısı:** Daemon thread'ler çıkış anında `finally` blokları veya dosya kapatma işlemleri tamamlanmadan aniden kesilebildiği için dosya yazma veya veritabanı işlemleri gibi yarım kalmaması gereken görevlerde kullanılmamalıdır.


In [ ]:
import threading
import time  # noqa: F811


def heartbeat() -> None:
    beat: int = 0
    while True:
        beat += 1
        print(f'Heartbeat #{beat}')
        time.sleep(0.15)


daemon_worker: threading.Thread = threading.Thread(target=heartbeat, daemon=True)
daemon_worker.start()

time.sleep(0.4)
print('Main thread finished; daemon thread will exit automatically.')


## 4. Limiting Concurrency with `threading.Semaphore`

`threading.Semaphore`, dahili bir sayaç kullanarak belirli bir kaynağa veya kod bloğuna aynı anda erişebilecek maksimum thread sayısını sınırlandıran senkronizasyon aracıdır[cite: 77].

**Syntax**
```python
sem = threading.Semaphore(value=3)
sem.acquire()
sem.release()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Kapasite Kontrolü:** `Lock` yalnızca 1 thread'e izin verirken, `Semaphore(n)` aynı anda en fazla `n` adet thread'in kaynağı kullanmasına izin verir[cite: 77]. Sayaç sıfıra indiğinde yeni gelen thread'ler, içerideki bir thread `release()` çağırana kadar kuyrukta bekler[cite: 77].
* **Kaynak Koruması:** Veritabanı bağlantı havuzlarını, dış API hız sınırlarını (rate limits) veya sınırlı donanım/dosya tanıtıcılarını aşırı yüklenmeden korumak için kullanılır[cite: 77].
* **`BoundedSemaphore`:** Yanlışlıkla `acquire()` sayısından fazla `release()` çağrılmasını önlemek için `threading.BoundedSemaphore(n)` kullanılabilir (fazla `release()` durumunda `ValueError` fırlatır).


In [ ]:
import threading
import time

pool_sem: threading.Semaphore = threading.Semaphore(2)


def api_request(req_id: int) -> None:
    pool_sem.acquire()
    try:
        print(f'Request {req_id} acquired slot')
        time.sleep(0.25)
        print(f'Request {req_id} releasing slot')
    finally:
        pool_sem.release()


threads: list[threading.Thread] = [
    threading.Thread(target=api_request, args=(i,)) for i in range(1, 6)
]

for t in threads:
    t.start()
for t in threads:
    t.join()


## 5. Context Manager Pattern (`with lock` & `with semaphore`)

`threading.Lock`, `RLock`, `Semaphore` ve `BoundedSemaphore` nesneleri bağlam yöneticisi (context manager) protokolünü destekler; `with` bloğu kullanıldığında blok girişinde otomatik `acquire()`, blok çıkışında ise (hata fırlatılsa dahi) otomatik `release()` çağrılır[cite: 78].

**Syntax**
```python
with lock:
    shared_resource.update()

with semaphore:
    limited_operation()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Hata Güvenliği:** Manuel `acquire()` / `release()` çiftlerinde istisna fırlatıldığında kilidin açık kalma (deadlock) riskini tamamen ortadan kaldırır[cite: 78].
* **Dar Kapsam İlkesi:** `with` bloğu yalnızca paylaşılan kaynağa erişilen kritik satırları kapsamalıdır; uzun G/Ç beklemelerini kilit içinde tutmak eşzamanlılık avantajını yok eder.


In [ ]:
import threading
import time

gate: threading.Semaphore = threading.Semaphore(3)


def bounded_task(task_id: int) -> None:
    with gate:
        print(f'Task {task_id} entered')
        time.sleep(0.2)
        print(f'Task {task_id} exited')


workers: list[threading.Thread] = [
    threading.Thread(target=bounded_task, args=(i,)) for i in range(1, 7)
]

for w in workers:
    w.start()
for w in workers:
    w.join()


## 6. Race Conditions & Thread Safety

Yarış durumu (race condition), iki veya daha fazla thread'in paylaşılan değiştirilebilir (mutable) bir veriyi senkronizasyon olmadan aynı anda okuyup değiştirmesi sonucunda, işlem sırasına bağlı olarak tutarsız ve öngörülemez sonuçlar üretilmesidir[cite: 79].

**Neden Oluşur ve Nasıl Önlenir?**
* **Bölünemez Olmayan (Non-Atomic) İşlemler:** `x += 1` veya `temp = value; temp += amount; value = temp` gibi işlemler tek bir adımda gerçekleşmez (oku -> hesapla -> yaz)[cite: 79]. İşletim sistemi okuma ile yazma arasında thread değişimi yaptığında güncellemelerden biri ezilir (lost update)[cite: 79].
* **GIL Yanılgısı:** GIL aynı anda tek bytecode çalışmasını sağlasa da birden fazla bytecode adımından oluşan okuma-güncelleme-yazma dizilerini atomik hale getirmez[cite: 74, 79].
* **Çözüm:** Paylaşılan değişkene yapılan tüm okuma-değiştirme-yazma adımlarını tek bir `with lock:` bloğu içine alarak işlemi atomik hale getirmek[cite: 79].


In [ ]:
import threading
import time

unsafe_balance: int = 0
safe_balance: int = 0
balance_lock: threading.Lock = threading.Lock()


def update_unsafe(delta: int, repeats: int) -> None:
    global unsafe_balance
    for _ in range(repeats):
        current: int = unsafe_balance
        time.sleep(0)
        unsafe_balance = current + delta


def update_safe(delta: int, repeats: int) -> None:
    global safe_balance
    for _ in range(repeats):
        with balance_lock:
            current: int = safe_balance
            time.sleep(0)
            safe_balance = current + delta


t_unsafe_add = threading.Thread(target=update_unsafe, args=(100, 10_000))
t_unsafe_sub = threading.Thread(target=update_unsafe, args=(-100, 10_000))
t_unsafe_add.start()
t_unsafe_sub.start()
t_unsafe_add.join()
t_unsafe_sub.join()

t_safe_add = threading.Thread(target=update_safe, args=(100, 10_000))
t_safe_sub = threading.Thread(target=update_safe, args=(-100, 10_000))
t_safe_add.start()
t_safe_sub.start()
t_safe_add.join()
t_safe_sub.join()

print(f'Unsafe balance (race condition): {unsafe_balance}')
print(f'Safe balance (with Lock): {safe_balance}')
